# 1. Setup

In [ ]:
# imports
import transformers
import torch
import logging
import random
import pandas as pd
from textstat import automated_readability_index
import json
import os

In [ ]:
# Basic vars
prompt = "Once upon a time, in a big forest, there lived a rhinoc"
model = transformers.AutoModelForCausalLM.from_pretrained('roneneldan/TinyStories-8M')
model.eval()
tokenizer = transformers.AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
input_ = tokenizer(prompt, return_tensors="pt")
prompt_token_length = input_['input_ids'].shape[1] #16

TORCH_SEED = 42
PY_SEED = 1234
NUM_COMPLETIONS = 20
MAX_GEN_TOKENS = 100
DATA_SAVE_DIR = os.path.join("data", "example")
os.makedirs(DATA_SAVE_DIR, exist_ok=True)
max_length = prompt_token_length + MAX_GEN_TOKENS # for batch size 1 this is a fixed quantity for all text = 116
transformers.set_seed(TORCH_SEED)
random.seed(PY_SEED)
logging.getLogger("transformers").setLevel(logging.ERROR)

choice_idxs = list(range(prompt_token_length, max_length)) # these are the positions that can be edited from in TPS. CHECK.

In [ ]:
def compute_ari(prompt, completions, model):
    # uncapped version for illustration purposes
    # paper uses custom py-readability implementation for edge case handling and cap=15
    aris = [automated_readability_index(prompt + c) for c in completions]
    return aris

def compute_log_probs(prompt, completions, model):
    # unbatched version for illustration purposes
    res = []

    for c in completions:

        full_text = prompt + c
        tmp = tokenizer(full_text, return_tensors="pt")
        next_token_logits = model(**tmp).logits
        next_token_log_probs = next_token_logits.log_softmax(dim=-1)
        current_token_log_probs = next_token_log_probs.roll(shifts=1, dims=-2) #align current token log probs with current tokens
        current_token_log_probs[:, 0, 0:prompt_token_length] = 0 #set all prompt token log probs to zero since these are given, P(prompt)=1
        chosen_token_ids = tmp['input_ids'] 
        chosen_current_token_log_probs = current_token_log_probs.gather(dim = -1, index=chosen_token_ids.unsqueeze(-1)).squeeze(-1) 
        joint_chosen_log_probs = chosen_current_token_log_probs.sum(dim=1)  # shape (1,)
        #to python float
        joint_chosen_log_probs = joint_chosen_log_probs.item()
        res.append(joint_chosen_log_probs)

    return res

def make_obs_fn(obs):
    if obs == 'ari':
        return compute_ari
    elif obs == 'log_probs':
        return compute_log_probs
    else:
        raise ValueError(f"Unsupported observable: {obs}. Supported: 'ari', 'log_probs'")

def generate_tokens(input_, model, tokenizer, prompt, max_length):
    output_tokens = model.generate(**input_, max_length=max_length, top_k=None, do_sample=True, temperature=1.0, 
                                    eos_token_id=None, pad_token_id=None)
    output_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    completion_text = output_text[len(prompt):]
    return output_tokens, completion_text

# 1. Direct Sampling Example

Setting `num_completions = 10^6` and adding ari-cap would give paper results (approx)

In [ ]:
def direct_sampling(input_, model, tokenizer, prompt, max_length, num_completions=10):

    completions = []

    with torch.no_grad():

        for _ in range(num_completions):
            output_tokens, completion_text = generate_tokens(input_, model, tokenizer, prompt, max_length)
            assert output_tokens.shape[0] == 1, f"Batch size greater than 1 not supported in this code snippet"
            assert output_tokens.shape[1] == max_length, f"Generated sequence length {output_tokens.shape[1]} does not match expected max_length {max_length}" #This might happen if stop-token is used
            completions.append(completion_text)

    aris = compute_ari(prompt, completions, None)
    log_probs = compute_log_probs(prompt, completions, model)

    return completions, aris, log_probs

In [ ]:
completions, aris, log_probs = direct_sampling(input_, model, tokenizer, prompt, max_length, num_completions=NUM_COMPLETIONS)

In [ ]:
completions[0:10]

In [ ]:
# quick histograms of both 
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].hist(aris, bins=20, color='blue', alpha=0.7)
axs[0].set_title("Automated Readability Indices")
axs[1].hist(log_probs, bins=20, color='green', alpha=0.7)
axs[1].set_title("Log Probabilities of Completions")
axs[0].set_xlabel("ARI")
axs[0].set_ylabel("Frequency")
axs[1].set_xlabel("Log Probability")
axs[1].set_ylabel("Frequency")
axs[0].set_yscale('log')
axs[1].set_yscale('log')

# 2. Biased Sampling

Running $10$ independent chains would reproduce paper

In [ ]:
def generate_proposal(current_tokens, model, tokenizer, prompt, max_length, choice_idxs):
    """Generates a proposal by randomly choosing an index from choice_idxs and generating tokens from the truncated input."""
    idx = random.choice(choice_idxs) #random edit point
    input_tokens_proposal = current_tokens.clone()[:, 0:idx]
    #make all all ones attention mask for proposal
    attention_mask_proposal = torch.ones_like(input_tokens_proposal) #Only works because no padding tokens in
    input_proposal = {'input_ids': input_tokens_proposal, 'attention_mask': attention_mask_proposal} #generate tokens expects dict-like input
    output_tokens_proposal, completion_text_proposal = generate_tokens(input_proposal, model, tokenizer, prompt, max_length)
    assert output_tokens_proposal.shape[0] == 1, f"Batch size greater than 1 not supported in this code snippet"
    assert output_tokens_proposal.shape[1] == max_length, f"Generated sequence length {output_tokens_proposal.shape[1]} does not match expected max_length {max_length}"
    return output_tokens_proposal, completion_text_proposal, idx

def compute_tps_acceptance_factor(bias, current_tokens, proposal_tokens, current_obs, proposal_obs, current_length, proposal_length):
    """Compute the acceptance factor according to Metropolis-Hastings criterion for TPS"""
    length_ratio = current_length / proposal_length
    obs_diff = proposal_obs - current_obs
    acceptance_factor = length_ratio * torch.exp(-bias * torch.tensor(obs_diff)).item() #check standard terminology
    acceptance_factor_min_1 = min(acceptance_factor, 1.0)
    return acceptance_factor_min_1, acceptance_factor, obs_diff, length_ratio

def tps_step(current_tokens, current_completion_text, model, tokenizer, prompt, max_length, choice_idxs, bias, compute_obs_fn, current_obs=None, current_length=None):
    """Performs a single TPS step: generates a proposal, computes observables, and acceptance factor.
    
    Expected signature for compute_obs_fn: compute_obs_fn(prompt, completion_text, model) -> observable_value
    
    """
    # Generate proposal
    proposal_tokens, proposal_completion_text, edit_idx = generate_proposal(current_tokens, model, tokenizer, prompt, max_length, choice_idxs)
    
    # Compute observables. Batch size 1 assumed throughout
    current_obs = compute_obs_fn(prompt, [current_completion_text], model)[0] if current_obs is None else current_obs
    proposal_obs = compute_obs_fn(prompt, [proposal_completion_text], model)[0]

    is_same_tokens = torch.equal(proposal_tokens, current_tokens)
    is_same_text = (proposal_completion_text == current_completion_text)
    is_same_obs = (proposal_obs == current_obs)

    # Compute acceptance factor
    current_length = current_tokens.shape[1]
    proposal_length = proposal_tokens.shape[1]
    alpha, acceptance_factor, obs_diff, length_ratio = compute_tps_acceptance_factor(bias, 
        current_tokens, proposal_tokens, current_obs, proposal_obs, current_length, proposal_length)

    step_info = {
        "is_same_tokens": is_same_tokens,
        "is_same_text": is_same_text,
        "is_same_obs": is_same_obs,
        "current_obs": current_obs,
        "proposal_obs": proposal_obs,
        "alpha": alpha,
        "acceptance_factor": acceptance_factor,
        "obs_diff": obs_diff,
        "length_ratio": length_ratio,
        "edit_idx": edit_idx
    }
    

    # Accept/reject step
    if random.random() < alpha:
        # Accept proposal
        accepted = 1
        step_info["accepted"] = accepted
        return proposal_tokens, proposal_completion_text,  proposal_obs, step_info
    else:
        # Reject proposal, keep current
        accepted = 0
        step_info["accepted"] = accepted
        return current_tokens, current_completion_text, current_obs, step_info

def biased_sampling(input_, model, tokenizer, prompt, max_length, choice_idxs, num_tps_steps_per_bias=5, biases=[0.1, 0.2], obs='ari'):
    """Performs biased sampling using annealed TPS to generate completions. Performs a single chain of num_tps_steps TPS steps."""

    def _get_bias_for_step(step):
        bias_idx = step // num_tps_steps_per_bias
        return biases[bias_idx]

    num_tps_steps = num_tps_steps_per_bias * len(biases)
    sampling_info = {"steps": []}
    observable_fn = make_obs_fn(obs)

    with torch.no_grad():
        # Initial generation and observable computation
        current_tokens, completion_text = generate_tokens(input_, model, tokenizer, prompt, max_length)
        current_length = current_tokens.shape[1]
        current_obs = None #will be computed in first step

        for step in range(num_tps_steps):
            bias = _get_bias_for_step(step)
            current_tokens, completion_text, current_obs, step_info = tps_step(current_tokens, 
                completion_text, model, tokenizer, prompt, max_length, choice_idxs, bias, 
                compute_obs_fn=observable_fn, current_obs=current_obs, current_length=current_length)
            # concat step info into sampling_info
            sampling_info["steps"].append(step)
            if step == 0: #init the keys of sampling_info
                for key in step_info.keys():
                    sampling_info[key] = []
            for key, value in step_info.items():
                sampling_info[key].append(value)

    # add fixed info
    sampling_info["bias"] = [_get_bias_for_step(step) for step in range(num_tps_steps)]
    sampling_info["observable"] = obs
    sampling_info["num_tps_steps_per_bias"] = num_tps_steps_per_bias
    sampling_info["num_tps_steps"] = num_tps_steps
    sampling_info["min_choice_idx"] = min(choice_idxs)
    sampling_info["max_choice_idx"] = max(choice_idxs)


    return current_tokens, completion_text, current_obs, sampling_info

In [ ]:
current_tokens, completion_text, current_obs, sampling_info = biased_sampling(input_, model, tokenizer, prompt, max_length, choice_idxs, num_tps_steps_per_bias=100, biases=[0.1, 0.2, 0.3], obs='ari')

In [ ]:
df = pd.DataFrame(sampling_info)
df

In [ ]:
# compute key statistics
acceptance_rates = df.groupby('bias')['accepted'].mean()
mean_obs = df.groupby('bias')['proposal_obs'].mean() #no burn-in applied
print("Acceptance Rate:", acceptance_rates.to_dict())
print("Mean Observable (no burn-in):", mean_obs.to_dict())

In [ ]:
# plot current_obs vs step highlighting bias changes
biases = df['bias'].unique()
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.plot(df['steps']+1, df['proposal_obs'], marker='o', linestyle='--', label='Proposal Observable', alpha=0.6)
plt.plot(df['steps'], df['current_obs'], marker='s', linestyle='-', label='Current Observable')
# cumulative mean current_obs for each bias region separately
cumulative_means = []
for bias in biases: #not sure this always works ascending with step
    bias_mask = df['bias'] == bias
    bias_obs = df.loc[bias_mask, 'current_obs']
    cummean = bias_obs.expanding().mean()
    cumulative_means.extend(cummean)
plt.plot(df['steps'], cumulative_means, color='red', marker='x', linestyle='--', label='Cumulative Mean Current Observable', linewidth=2)
# highlight bias regions
for bias in biases:
    bias_steps = df[df['bias'] == bias]['steps']
    plt.axvspan(bias_steps.min(), bias_steps.max(), alpha=0.2)
plt.xlabel('TPS Step')
plt.ylabel('Proposal Observable (ARI)')
plt.title('Proposal Observable vs TPS Step with Bias Highlighting')
plt.legend()
plt.show()

In [ ]:
# Extract data to save for reweighting and analysis scripts. 

"""Parameters to recreate Figure 2a in the paper are commented out below."""
# steps_per_bias = 40000
# biases = [[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
#           [-0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1.0]]
# num_trajectories = 10

"""Small example."""
steps_per_bias = 10
both_obs_biases = {"ari": [[0.1, 0.2, 0.3], [-0.1, -0.2, -0.3]],
              "log_probs": [[[0.01, 0.02, 0.03], [-0.01, -0.02, -0.03]]]}

num_trajectories = 3

for obs_name in ["ari", "log_probs"]:
    all_trajectories = []
    all_biases = both_obs_biases[obs_name]
    for biases in all_biases:
        trajs_per_biases = []
        for num_traj in range(num_trajectories):
            _, _, _, sampling_info = biased_sampling(input_, model, tokenizer, prompt, max_length, choice_idxs, num_tps_steps_per_bias=steps_per_bias, biases=biases, obs=obs_name)
            trajectory = sampling_info["current_obs"]
            trajs_per_biases.append(trajectory)
        all_trajectories.append(trajs_per_biases)

    with open(os.path.join(DATA_SAVE_DIR, f"{obs_name}_trajectories.json"), "w") as f:
        json.dump([all_trajectories, all_biases, steps_per_bias], f)

In [ ]:
"""Parameters to recreate the orange curve in Figure 3b are commented out below."""
# num_unbiased_samples = 4200000

num_unbiased_samples = 50

_, aris_save, log_probs_save = direct_sampling(input_, model, tokenizer, prompt, max_length, num_completions=num_unbiased_samples)

with open(os.path.join(DATA_SAVE_DIR, "ari_unbiased_samples.json"), "w") as f:
    json.dump(aris_save, f)
    
with open(os.path.join(DATA_SAVE_DIR, "log_probs_unbiased_samples.json"), "w") as f:
    json.dump(log_probs_save, f)